In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalog", "insurance_project")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text(
    "source_path",
    "/Volumes/insurance_project/landing/raw_files/initial"
)

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
source_path = dbutils.widgets.get("source_path").rstrip("/")

for value in [catalog, bronze_schema]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid catalog/schema name: {value}")

expected_counts = {
    "branches": 8,
    "insurance_products": 10,
    "agents": 20,
    "customers": 100000,
    "policies": 100000,
    "premium_payments": 100000,
    "beneficiaries": 100000,
    "claims": 100000,
    "claim_assessments": 100000,
    "claim_settlements": 100000
}

print("Source:", source_path)
print("Target:", f"{catalog}.{bronze_schema}")


In [0]:
audit_rows = []

for table_name, expected_count in expected_counts.items():
    file_path = f"{source_path}/{table_name}.csv"
    target_table = f"{catalog}.{bronze_schema}.{table_name}"

    print(f"Loading {file_path}")

    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .load(file_path)
    )

    bronze_df = (
        source_df
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file")
        )
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_system", F.lit("insurance_csv"))
    )

    actual_count = bronze_df.count()

    if actual_count != expected_count:
        raise ValueError(
            f"{table_name}: expected {expected_count}, "
            f"received {actual_count}"
        )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    audit_rows.append(
        (table_name, actual_count, target_table)
    )

display(
    spark.createDataFrame(
        audit_rows,
        ["table_name", "row_count", "target_table"]
    )
)


In [0]:
customers = spark.table(
    f"{catalog}.{bronze_schema}.customers"
)

customers.printSchema()
display(customers.limit(10))
